# 05 — Evaluation, threshold analysis, risk scores

## Confusion matrix language

| | Predicted legit | Predicted fraud |
|---|---|---|
| **Actual legit** | TN | FP |
| **Actual fraud** | FN | TP |

- **TP** — fraud we caught.
- **TN** — legitimate traffic we left alone.
- **FP** — false alarm (analyst time, blocked customer).
- **FN** — missed fraud (**usually the most expensive** in payments: the money is already gone).

We do **not** pick a "best" model from accuracy.

## Precision

Of rows we flagged as fraud, what fraction were truly fraud?

High precision → fewer wasted reviews. Too high a precision target often means we hide in conservative thresholds and miss fraud.

## Recall

Of *all* actual fraud, what fraction did we catch?

High recall → fewer FN. The cost is more FP.

## F1

Harmonic mean of precision and recall. Useful when you want a single compromise, not when one error type is 100× more expensive than the other.

## ROC-AUC

Area under TPR vs FPR. Can look optimistic on rare events because many TN make FPR tiny.

## PR-AUC (average precision)

Area under precision vs recall. The **no-skill baseline is the fraud prevalence** (~0.002), not 0.5. That is why PR-AUC is the headline ranking metric in this repo.

## Thresholds

`0.5` is a convenience, not a business law. Lower threshold → more recall, more FP. The app exposes this as policy.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_processing import dataset_overview, drop_duplicates_if_present, load_raw_dataset
from src.utils import RAW_DATASET_PATH

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.evaluate import classification_metrics, threshold_sweep

payload_path = ROOT / "models" / "eval_payload.json"
meta_path = ROOT / "models" / "model_metadata.json"
assert payload_path.exists(), "Run python -m src.train first"
payload = json.loads(payload_path.read_text())
y_true = np.array(payload["y_true"])
y_prob = np.array(payload["y_prob"])
metrics = classification_metrics(y_true, y_prob, threshold=0.5)
metrics


In [ ]:
sweep = threshold_sweep(y_true, y_prob)
sweep


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep.threshold, sweep.precision, marker="o", label="precision")
ax.plot(sweep.threshold, sweep.recall, marker="o", label="recall")
ax.plot(sweep.threshold, sweep.f1, marker="o", label="f1")
ax.set_xlabel("threshold")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Threshold policy curve")


## Risk score vs probability vs category

- **Model probability:** `predict_proba` output in \([0,1]\). Not automatically a calibrated long-run frequency.
- **Risk score:** `round(100 * probability)` — a communication scale.
- **Category:** LOW (0–30), MEDIUM (31–70), HIGH (71–100) — **policy bands**.

`predict_risk(transaction)` in `src/predict.py` returns all four of: probability, score, category, binary prediction.


In [ ]:
from src.predict import probability_to_risk_score, risk_category
for p in [0.05, 0.4, 0.87]:
    s = probability_to_risk_score(p)
    print(p, s, risk_category(s))


Open `reports/model_report.md` for the full comparison table produced on your machine. Never paste invented metrics into a CV.
